In [11]:
import jax
from flax import nnx
import jax.numpy as jnp
import optax

In [12]:
# model simple vector field network v_theta(x, t)
class HiddenNet(nnx.Module):
    def __init__(self, dim, *, rngs: nnx.Rngs) -> None:
        self.linear = nnx.Linear(dim, dim, rngs=rngs)

    def __call__(self, x):
        return nnx.softplus(self.linear(x))

class VectorFieldNetwork(nnx.Module):

    def __init__(self, input_dim=2, hidden_dim=64, num_hidden_layers=5, *, rngs: nnx.Rngs) -> None:
        self.linear1 = nnx.Linear(input_dim + 1, hidden_dim, rngs=rngs)

        @nnx.split_rngs(splits=num_hidden_layers)
        @nnx.vmap(in_axes=0, out_axes=0)
        def create_hidden_layer(rngs):
            return HiddenNet(hidden_dim, rngs=rngs)

        self.hidden_layers = create_hidden_layer(rngs)
        self.linear3 = nnx.Linear(hidden_dim, input_dim, rngs=rngs)

    def __call__(self, x, t):
        # x [batch_size, input_dim]
        # t [batch_size, 1]
        inputs = jnp.concatenate([x, t], axis=-1) # [batch_size, input_dim + 1]
        out1 = nnx.softplus(self.linear1(inputs))

        @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=nnx.Carry)
        def apply_hidden_layers(x, layer):
            x = layer(x)
            return x

        h = apply_hidden_layers(out1, self.hidden_layers)
        return self.linear3(h) # [batch_size, input_dim]

In [13]:
# 2. Helper to generate toy 2D target data (a circle)
def sample_target_data(batch_size, rng_key):
    k1, k2 = jax.random.split(rng_key)
    theta = jax.random.uniform(k1, (batch_size, 1)) * 2 * jnp.pi
    r = 2.0 + jax.random.uniform(k2, (batch_size, 1)) * 0.1
    x1 = jnp.concatenate([r * jnp.cos(theta), r * jnp.sin(theta)], axis=-1)  # [batch_size, 2]
    return x1

print(sample_target_data(5, jax.random.key(0)).shape)


(5, 2)


In [14]:
@nnx.jit
def train_step(model, optimizer, xt, t, u_t):
    def loss_fn(model):
        v_pred = model(xt, t)  # [batch_size, 2]
        return jnp.mean((v_pred - u_t) ** 2)

    loss, grads = jax.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss



def train_flow_matching(epochs=1000, batch_size=128, learning_rate=1e-3):
    model = VectorFieldNetwork(input_dim=2, hidden_dim=128, rngs=nnx.Rngs(0))
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)

    print("Starting training flow matching...")

    for epoch in range(epochs):
        epoch_key = jax.random.key(epoch)
        k1, k2, k3 = jax.random.split(epoch_key, num=3)

        # sample endpoint data: x1 ~ target distribution, x0 ~ simple prior (e.g., Gaussian)
        x1 = sample_target_data(batch_size, k1)
        x0 = jax.random.normal(k2, (batch_size, 2))  # [batch_size, 2]

        # sample random time t ~ Uniform(0, 1)
        t = jax.random.uniform(k3, (batch_size, 1)) # [batch_size, 1]

        # Construct the conditional path (Linear interpolation / Optimal Transport)
        xt = (1 - t) * x0 + t * x1  # [batch_size, 2]

        # Target velocity field u_t(x | x_0, x_1) = x_1 - x_0
        u_t = x1 - x0  # [batch_size, 2]

        loss = train_step(model, optimizer, xt, t, u_t)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")

    return model


In [15]:
# Sampling / Inference Function (Solving the ODE via Euler Integration)
@jax.jit(static_argnames=["num_samples", "num_steps"])
def sample_from_model(model, num_samples=1000, num_steps=50):
    model.eval()
    key = jax.random.key(42)
    # sample initial pure noise from simple prior x_0 ~ N(0, I)
    xt = jax.random.normal(key, (num_samples, 2))
    dt = 1.0 / num_steps

    def euler_step(i, carry):
        xt, model = carry
        t_val = i * dt
        t = jnp.full((num_samples, 1), t_val) # [num_samples, 1]

        vt = model(xt, t)  # [num_samples, 2]
        xt = xt + vt * dt
        return (xt , model)

    xt, _ = jax.lax.fori_loop(jnp.int32(0), num_steps,euler_step , (xt, model))

    return xt


In [ ]:
trained_model = train_flow_matching(epochs=1001, batch_size=256, learning_rate=1e-3)


Starting training flow matching...
Epoch 0, Loss: 3.8360
Epoch 100, Loss: 2.8764


In [ ]:

# Sample from the trained model
generated_samples = sample_from_model(trained_model, num_samples=5, num_steps=50)
print("\nGenerated 2D coordinates on the learned circle:")
print(generated_samples)


Generated 2D coordinates on the learned circle:
[[ 0.26678634  1.9040599 ]
 [ 1.839187    0.46235162]
 [-0.8044956   1.7033975 ]
 [-1.8059622   1.0032756 ]
 [ 1.330911    1.4711717 ]]


In [ ]:
# Mean flows - https://arxiv.org/pdf/2412.06264 . get average speed and allow for 1 step inference.
class HiddenFlow(nnx.Module):
    def __init__(self, dim, *, rngs: nnx.Rngs) -> None:
        self.linear = nnx.Linear(dim, dim, rngs=rngs)

    def __call__(self, x):
        return nnx.silu(self.linear(x))

class MeanFlowNetwork(nnx.Module):

    def __init__(self, input_dim=2, hidden_dim=64, num_hidden_layers=5, *, rngs: nnx.Rngs) -> None:
        # Takes x, r, and t. Dimension = input_dim + 2
        self.linear1 = nnx.Linear(input_dim + 2, hidden_dim, rngs=rngs)
        self.hidden = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs)

        # @nnx.split_rngs(splits=num_hidden_layers)
        # @nnx.vmap(in_axes=0, out_axes=0)
        # def create_hidden_layer(rngs):
        #     return HiddenFlow(hidden_dim, rngs=rngs)

        # self.hidden_layers = create_hidden_layer(rngs)
        self.hidden_layers = nnx.List([HiddenFlow(hidden_dim, rngs=rngs) for _ in range(num_hidden_layers)])
        self.linear2 = nnx.Linear(hidden_dim, input_dim, rngs=rngs)

    def __call__(self, x, r, t):
        # x [batch_size, input_dim]
        # r [batch_size, 1]
        # t [batch_size, 1]
        inputs = jnp.concatenate([x, r, t], axis=-1) # [batch_size, input_dim + 2]
        out = nnx.silu(self.linear1(inputs))

        for layer in self.hidden_layers:
            out = layer(out)

        # @nnx.scan(in_axes=(nnx.Carry, 0), out_axes=nnx.Carry)
        # def apply_hidden_layers(x, layer):
        #     x = layer(x)
        #     return x

        # h = apply_hidden_layers(out1, self.hidden_layers)

        return self.linear2(out) # [batch_size, input_dim]

In [ ]:
@nnx.jit
def train_step_mean_flow(model, optimizer, xt, r, t, u):
    def loss_fn(model):
        # Tangent vectors (the direction of changes with respect to dx/dt, dr/dt, dt/dt)
        # dx/dt along the path is exactly the velocity vector 'u'.
        # dr/dt = 0 (holding the start interval steady)
        # dt/dt = 1 (advancing the target clock)
        tangent_x = u
        tangent_r = jnp.zeros_like(r)
        tangent_t = jnp.ones_like(t)

        # Compute the network's prediction (v_pred) and its derivative along the path (dvdt)
        v_pred, dvdt = jax.jvp(lambda x, r_in, t_in: model(x, r_in, t_in), (xt, r, t),(tangent_x, tangent_r, tangent_t))

        # 4. The MeanFlow Identity targets:
        # The average velocity target relies on the current predictions and their derivatives
        u_target = u - (t - r) * dvdt

        # 5. Apply a stop gradient to the target side (just like in Reinforcement Learning or EMA targets)
        return jnp.mean((v_pred - jax.lax.stop_gradient(u_target)) ** 2)


    loss, grads = jax.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

def train_mean_flow_matching(epochs=1000, batch_size=128, learning_rate=1e-3):
    model = MeanFlowNetwork(input_dim=2, hidden_dim=128, rngs=nnx.Rngs(0))
    optimizer = nnx.Optimizer(model, optax.adam(learning_rate), wrt=nnx.Param)

    print("Starting training mean flow matching...")

    for epoch in range(epochs):
        epoch_key = jax.random.key(epoch)
        k1, k2, k3, k4 = jax.random.split(epoch_key, num=4)

        # sample endpoint data: x1 ~ target distribution, x0 ~ simple prior (e.g., Gaussian)
        x1 = sample_target_data(batch_size, k1)
        x0 = jax.random.normal(k2, (batch_size, 2))

        # sample 2 2 time points r,t such that 0 <= r < t <= 1
        r = jax.random.uniform(k3, (batch_size, 1))
        t = r + jax.random.uniform(k4, (batch_size, 1)) * (1.0 - r)

        # Construct the conditional path (Linear interpolation / Optimal Transport)
        xt = (1 - t) * x0 + t * x1
        u = x1 - x0  # original instantaneous velocity field

        loss = train_step_mean_flow(model, optimizer, xt, r, t, u)

        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {loss:.4f}")
    return model


In [ ]:
trained_mean_flow_model = train_mean_flow_matching(epochs=1001, batch_size=256, learning_rate=1e-3)

Starting training mean flow matching...
x shape: (256, 2), r shape: (256, 1), t shape: (256, 1)
Epoch 0, Loss: 3.0794
Epoch 100, Loss: 2.3068
Epoch 200, Loss: 2.7383
Epoch 300, Loss: 3.1631
Epoch 400, Loss: 3.7538
Epoch 500, Loss: 3.7370
Epoch 600, Loss: 3.6407
Epoch 700, Loss: 3.1606
Epoch 800, Loss: 2.9221
Epoch 900, Loss: 3.1755
Epoch 1000, Loss: 5.0493


In [ ]:
def mean_flow_sample(model, num_samples):
    model.eval()
    key = jax.random.key(42)
    # sample initial pure noise from simple prior x_0 ~ N(0, I)
    x0 = jax.random.normal(key, (num_samples, 2))

    # define absoulte interval boundaries
    r = jnp.zeros((num_samples, 1))
    t = jnp.ones((num_samples, 1))

    # 3. Model outputs the true average velocity required to cross the whole trajectory
    average_velocity = model(x0, r, t)

    # 4. Single step integration: x_1 = x_0 + average_velocity * (t - r)
    # Since (t - r) = (1 - 0) = 1, it simplifies to:
    x1 = x0 + average_velocity

    return x1


In [ ]:
# Sample from the trained model
generated_samples = mean_flow_sample(trained_mean_flow_model, num_samples=5)
print("\nGenerated 2D coordinates on the learned circle:")
print(generated_samples)

x shape: (5, 2), r shape: (5, 1), t shape: (5, 1)

Generated 2D coordinates on the learned circle:
[[-0.3838959   0.6198036 ]
 [ 0.5565535  -0.27457172]
 [-0.6684343  -0.01444781]
 [-3.7769747   1.3555549 ]
 [ 1.0909503   1.5317404 ]]


## Normalizing flows example
1D affine flow. learns $x = e^{log s}z + b$ (scale + shift) where $z \sim \mathcal{N}(0, 1)$ and $s, b$ are learnable parameters. The log-determinant of the Jacobian is $\log |s|$. 

```python

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx
import optax

class Affine1DFlow(nnx.Module):
    def __init__(self):
        # start close to identity transformation - x = 1*z + 0
        self.log_s = nnx.Param(jnp.zeros(()))
        self.b = nnx.Param(jnp.zeros(()))

    def inverse(self, x):
        s = jnp.exp(self.log_s)
        # inverse: z = (x - b) / s
        z = (x - self.b) / s

        # dz/dx = 1/s, so log|dz/dx| = -log|s|
        return z, -self.log_s

    def forward(self, z):
        # x = e^{log s} * z + b
        s = jnp.exp(self.log_s)
        return s * z + self.b

def standard_normal_logpdf(z):
    # log N(z; 0, 1)
    return -0.5 * (z ** 2 + jnp.log(2.0 * jnp.pi)) # or use jax.scipy.stats.norm.logpdf(z)

def negative_log_likelihood(model, x):
    """
    Flow likelihood:

        log p_X(x) = log p_Z(z) + log |dz/dx|

    where z = f^{-1}(x).
    """
    z, log_det = model.inverse(x)

    log_px = standard_normal_logpdf(z) + log_det
    return -jnp.mean(log_px)


In [ ]:
# -------------------------
# Make toy data
# -------------------------

key = jax.random.key(0)

# True data distribution:
# x = 2.5 * z + 4.0
true_z = jax.random.normal(key, shape=(2048,))
x_train = 2.5 * true_z + 4.0

In [ ]:
# train flow

@nnx.jit
def train_step_1(model, optimizer, x):
    def loss_fn(model):
        return negative_log_likelihood(model, x)

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model, grads)
    return loss

model = Affine1DFlow()
optimizer = nnx.Optimizer(model, optax.adam(1e-3), wrt=nnx.Param)

for step in range(10000):
    loss = train_step_1(model, optimizer, x_train)
    if step % 1000 == 0:
        print(
            step,
            "loss =", float(loss),
            "scale =", float(jnp.exp(model.log_s.get_value())),
            "shift =", float(model.b.get_value()),
        )

0 loss = 11.915657043457031 scale = 1.0010005235671997 shift = 0.0009999932954087853
1000 loss = 3.9108431339263916 scale = 1.9053137302398682 shift = 0.6878643035888672
2000 loss = 2.9353084564208984 scale = 2.606222629547119 shift = 1.1232705116271973
3000 loss = 2.6947946548461914 scale = 3.0749473571777344 shift = 1.5091594457626343
4000 loss = 2.594180107116699 scale = 3.2410075664520264 shift = 1.9223135709762573
5000 loss = 2.5043954849243164 scale = 3.095919609069824 shift = 2.4158546924591064
6000 loss = 2.413358211517334 scale = 2.7938857078552246 shift = 2.991124391555786
7000 loss = 2.3587093353271484 scale = 2.582021713256836 shift = 3.5220706462860107
8000 loss = 2.3452048301696777 scale = 2.526664972305298 shift = 3.8395869731903076
9000 loss = 2.344193696975708 scale = 2.5225324630737305 shift = 3.940948486328125


In [ ]:
# sample from trained flow
z_sample = jax.random.normal(jax.random.key(42), shape=(5,))
x_sample = model.forward(z_sample)

true_samples = 2.5 * z_sample + 4.0
print("\nGenerated samples:", x_sample)
assert jnp.allclose(x_sample, true_samples, atol=0.5), "Generated samples are not close to true samples!"

print("Learned scale:", jnp.exp(model.log_s.get_value()))
print("Learned shift:", model.b.get_value())
# see that learned scale is close to 2.5 and learned shift is close to 4.0, which matches the true data distribution.


Initial samples from standard normal: [-0.02830462  0.46713185  0.29570296  0.15354592 -0.12403282]
True samples: [3.9292386 5.1678295 4.7392573 4.383865  3.689918 ]

Generated samples: [3.8818295 5.1315603 4.6991334 4.3405447 3.6403568]
Learned scale: 2.5224838
Learned shift: 3.9532275


# Tarflow - https://arxiv.org/abs/2412.06329

In [ ]:
import math
from dataclasses import dataclass

import jax
import jax.numpy as jnp
from flax import nnx

@dataclass(frozen=True)
class TarFlowConfig:
    img_size: int = 64          # H = W
    in_channels: int = 3        # C
    patch_size: int = 2         # S
    model_dim: int = 768        # M, the transformer width
    n_heads: int = 12
    n_flows: int = 8            # T
    n_layers: int = 8           # K, per flow block
    num_classes: int = 1000     # class-conditional; use 1 for unconditional
    noise_std: float = 0.05     # sigma, section 2.4
    dtype: jnp.dtype = jnp.float32

    @property
    def seq_len(self) -> int:
        """N = H * W / S**2."""
        return (self.img_size // self.patch_size) ** 2

    @property
    def token_dim(self) -> int:
        """D = C * S**2."""
        return self.in_channels * self.patch_size**2

# patchify and unpatchify functions for image data, converting between image tensors and patch sequences.

def patchify(x: jax.Array, patch_size: int) -> jax.Array:
    """(B, C, H, W) -> (B, N, D) with N = H*W/S**2, D = C*S**2."""
    B, C, H, W = x.shape
    S = patch_size
    x = x.reshape(B, C, H // S, S, W // S, S)      # (B, C, H/S, S, W/S, S)
    x = x.transpose(0, 2, 4, 1, 3, 5)              # (B, H/S, W/S, C, S, S)
    return x.reshape(B, (H // S) * (W // S), C * S * S)   # (B, N, D)


def unpatchify(z: jax.Array, patch_size: int, channels: int) -> jax.Array:
    """(B, N, D) -> (B, C, H, W). Exact inverse of `patchify`."""
    B, N, D = z.shape
    S, C = patch_size, channels
    grid = int(math.isqrt(N))                      # H/S == W/S
    z = z.reshape(B, grid, grid, C, S, S)          # (B, H/S, W/S, C, S, S)
    z = z.transpose(0, 3, 1, 4, 2, 5)              # (B, C, H/S, S, W/S, S)
    return z.reshape(B, C, grid * S, grid * S)     # (B, C, H, W)


In [ ]:
class CausalSelfAttention(nnx.Module):
    """Standard causal MHA.

    Two modes:
      * parallel  (pos is None): full (L, L) causal mask. Used at TRAINING time,
        where the whole sequence is known — this is the cheap direction.
      * decode    (pos is an int/traced scalar): one token at a time, reading and
        writing a KV cache. Used at SAMPLING time, where token i cannot be
        computed until token i-1 exists.

    That asymmetry is the whole practical story of TarFlow: training is one
    parallel pass, sampling is N sequential steps per flow block.
    """

    def __init__(self, dim: int, n_heads: int, max_len: int, *, rngs: nnx.Rngs):
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.max_len = max_len  # N + 1 (conditioning token)
        self.scale = self.head_dim**-0.5

        self.qkv = nnx.Linear(dim, 3 * dim, use_bias=False, rngs=rngs)
        self.proj = nnx.Linear(dim, dim, use_bias=False, rngs=rngs)

        # Populated by `init_cache`; None during training.
        self.cache_k: nnx.Variable | None = None
        self.cache_v: nnx.Variable | None = None

    def init_cache(self, batch: int, dtype: jnp.dtype) -> None:
        shape = (batch, self.n_heads, self.max_len, self.head_dim)  # (B, Hh, L, hd)
        self.cache_k = nnx.Variable(jnp.zeros(shape, dtype))
        self.cache_v = nnx.Variable(jnp.zeros(shape, dtype))

    def _split_heads(self, t: jax.Array) -> jax.Array:
        """(B, L, M) -> (B, n_heads, L, head_dim)."""
        B, L, _ = t.shape
        return t.reshape(B, L, self.n_heads, self.head_dim).transpose(0, 2, 1, 3)

    def __call__(self, h: jax.Array, pos: jax.Array | int | None = None) -> jax.Array:
        # h: (B, L, M).  L == N+1 when parallel, L == 1 when decoding.
        B, L, M = h.shape
        q, k, v = jnp.split(self.qkv(h), 3, axis=-1)  # 3 x (B, L, M)
        q = self._split_heads(q)  # (B, Hh, L, hd)
        k = self._split_heads(k)
        v = self._split_heads(v)

        if pos is None:
            # --- parallel path (training) -----------------------------------
            # mask[i, j] = True iff j <= i
            idx = jnp.arange(L)
            mask = idx[:, None] >= idx[None, :]  # (L, L)
            mask = mask[None, None, :, :]  # (1, 1, L, L)
        else:
            # --- decode path (sampling) -------------------------------------
            assert L == 1, "decode consumes exactly one token at a time"
            assert self.cache_k is not None and self.cache_v is not None, "call init_cache() first"
            # Write this token's K/V into slot `pos` of the cache.
            self.cache_k.value = self.cache_k.value.at[:, :, pos, :].set(k[:, :, 0, :])
            self.cache_v.value = self.cache_v.value.at[:, :, pos, :].set(v[:, :, 0, :])
            k = self.cache_k.value  # (B, Hh, max_len, hd)
            v = self.cache_v.value
            # Attend to slots 0..pos only; the rest of the cache is stale zeros.
            mask = (jnp.arange(self.max_len) <= pos)[None, None, None, :]

        attn: jax.Array = jnp.einsum("bhqd,bhkd->bhqk", q, k) * self.scale  # (B, Hh, L, Lk)
        attn = jnp.where(mask, attn, jnp.finfo(attn.dtype).min)
        attn = jax.nn.softmax(attn.astype(jnp.float32), axis=-1).astype(h.dtype)

        out = jnp.einsum("bhqk,bhkd->bhqd", attn, v)  # (B, Hh, L, hd)
        out = out.transpose(0, 2, 1, 3).reshape(B, L, M)  # (B, L, M)
        return self.proj(out)

class MLP(nnx.Module):
    def __init__(self, dim: int, expansion: int = 4, *, rngs: nnx.Rngs):
        self.fc1 = nnx.Linear(dim, expansion * dim, rngs=rngs)
        self.fc2 = nnx.Linear(expansion * dim, dim, rngs=rngs)

    def __call__(self, h: jax.Array) -> jax.Array:  # (B, L, M) -> (B, L, M)
        return self.fc2(jax.nn.gelu(self.fc1(h), approximate=True))

class TransformerLayer(nnx.Module):
    """Pre-LN transformer layer. One of the K layers inside a flow block."""

    def __init__(self, dim: int, n_heads: int, max_len: int, *, rngs: nnx.Rngs):
        self.norm1 = nnx.LayerNorm(dim, rngs=rngs)
        self.attn = CausalSelfAttention(dim, n_heads, max_len, rngs=rngs)
        self.norm2 = nnx.LayerNorm(dim, rngs=rngs)
        self.mlp = MLP(dim, rngs=rngs)

    def __call__(self, h: jax.Array, pos=None) -> jax.Array:
        h = h + self.attn(self.norm1(h), pos=pos)
        h = h + self.mlp(self.norm2(h))
        return h

In [ ]:

class ARFlowBlock(nnx.Module):
    """A single autoregressive flow block, i.e. one step t of T.

    Forward (data -> latent), paper eq. 3:

        z~^t     = pi^t(z^t)
        z^{t+1}_i = z~^t_i                                             if i = 0
                  = (z~^t_i - mu^t_i(z~^t_<i)) * exp(-a^t_i(z~^t_<i))  if i > 0

    Inverse (latent -> data), paper eq. 4:

        z~^t_i = z^{t+1}_i                                             if i = 0
               = z^{t+1}_i * exp(a^t_i(z~^t_<i)) + mu^t_i(z~^t_<i)     if i > 0
        z^t    = (pi^t)^-1(z~^t)

    Log-determinant, paper eq. 5:

        log|det dz^{t+1}/dz^t| = - sum_{i=1}^{N-1} sum_{j=0}^{D-1} a^t_i[j]

    The Jacobian is triangular because token i only depends on tokens < i, so
    its determinant is the product of the diagonal entries exp(-a), and the
    log-det collapses to a plain sum over the network's own outputs. That is
    *why* the architecture has to be autoregressive: causality is what buys the
    tractable determinant, not a modelling preference.
    """

    def __init__(self, cfg: TarFlowConfig, flow_index: int, *, rngs: nnx.Rngs):
        self.cfg = cfg
        # pi^t: reverse the sequence on every other block so that information
        # can flow in both directions across the stack (section 2.3).
        self.reverse = bool(flow_index % 2)

        N, D, M = cfg.seq_len, cfg.token_dim, cfg.model_dim

        self.proj_in = nnx.Linear(D, M, rngs=rngs)  # D -> M
        self.pos_embed = nnx.Param(  # (1, N+1, M)
            0.02 * jax.random.normal(rngs.params(), (1, N + 1, M))
        )
        # num_classes + 1: the extra index is the "null" label used for the
        # unconditional branch of classifier-free guidance (eq. 10).
        self.class_embed = nnx.Embed(cfg.num_classes + 1, M, rngs=rngs)

        self.layers = [
            TransformerLayer(M, cfg.n_heads, max_len=N + 1, rngs=rngs)
            for _ in range(cfg.n_layers)  # K layers
        ]
        self.norm_out = nnx.LayerNorm(M, rngs=rngs)

        # Zero-init => mu = 0 and a = 0 => exp(-a) = 1 => this block is exactly
        # the identity at initialisation. With T*K = 64 layers, a stack that is
        # not the identity at init does not train.
        self.head = nnx.Linear(
            M,
            2 * D,  # -> [mu | a]
            kernel_init=nnx.initializers.zeros_init(),
            bias_init=nnx.initializers.zeros_init(),
            rngs=rngs,
        )

    # -- permutation pi^t ---------------------------------------------------

    def _permute(self, z: jax.Array) -> jax.Array:
        return z[:, ::-1] if self.reverse else z  # reversal is self-inverse

    _unpermute = _permute

    # -- parallel parameter prediction (training direction) -----------------

    def _params(self, z: jax.Array, y: jax.Array) -> tuple[jax.Array, jax.Array]:
        """z: (B, N, D) already permuted, y: (B,) labels -> mu, a each (B, N, D).

        The conditioning token is PREPENDED, which handles the strict-inequality
        shift for free: with the extended sequence

            [c, z_0, z_1, ..., z_{N-1}]        length N+1

        a causal transformer output at extended position i has seen exactly
        [c, z_0, ..., z_{i-1}] = z_<i plus the label. So output slice [0:N]
        gives the parameters for tokens 0..N-1 with the correct dependency.

        Getting this wrong is the classic TarFlow bug: an off-by-one that leaks
        z_i into mu_i makes the map non-invertible, and the training loss still
        descends beautifully because training is teacher-forced. You only find
        out when sampling produces noise.
        """
        B, N, D = z.shape
        h = self.proj_in(z)  # (B, N, M)
        c = self.class_embed(y)[:, None, :]  # (B, 1, M)
        h = jnp.concatenate([c, h], axis=1)  # (B, N+1, M)
        h = h + self.pos_embed.value  # (B, N+1, M)

        for layer in self.layers:  # K layers
            h = layer(h, pos=None)  # parallel path
        h = self.norm_out(h)

        params = self.head(h[:, :N])  # (B, N, 2D)
        mu, a = jnp.split(params, 2, axis=-1)  # (B, N, D) each

        # Paper eq. 3 leaves token 0 untransformed. Zero its parameters so the
        # map is literally the identity there.
        # (The reference implementation instead lets token 0 be an affine
        #  function of the class token alone — still invertible, slightly more
        #  expressive. Flip this if you want that variant.)
        zero = jnp.zeros_like(mu[:, :1])
        mu = jnp.concatenate([zero, mu[:, 1:]], axis=1)  # (B, N, D)
        a = jnp.concatenate([zero, a[:, 1:]], axis=1)  # (B, N, D)
        return mu, a

    # -- forward: data -> latent (eq. 3) ------------------------------------

    def forward(self, z: jax.Array, y: jax.Array) -> tuple[jax.Array, jax.Array]:
        """z: (B, N, D) -> (z_next: (B, N, D), logdet: (B,)).

        Fully parallel: one transformer pass for the whole sequence.
        """
        z = self._permute(z)  # z~^t
        mu, a = self._params(z, y)  # (B, N, D) each
        z_next = (z - mu) * jnp.exp(-a)  # eq. 3
        logdet = -a.sum(axis=(1, 2))  # eq. 5, (B,)
        return z_next, logdet

    # -- inverse, naive: latent -> data (eq. 4) -----------------------------

    def inverse_naive(self, z_next: jax.Array, y: jax.Array) -> jax.Array:
        """O(N^2 * K). Literal transcription of eq. 4 — use it to check the
        cached version, not to generate samples."""
        B, N, D = z_next.shape
        z = jnp.zeros_like(z_next).at[:, 0].set(z_next[:, 0])  # token 0: identity
        for i in range(1, N):
            mu, a = self._params(z, y)  # recompute all
            z = z.at[:, i].set(z_next[:, i] * jnp.exp(a[:, i]) + mu[:, i])
        return self._unpermute(z)

    # -- inverse, KV-cached: the one you actually sample with ---------------

    def init_cache(self, batch: int) -> None:
        for layer in self.layers:
            layer.attn.init_cache(batch, self.cfg.dtype)

    def _decode_step(self, token: jax.Array, pos: int) -> tuple[jax.Array, jax.Array]:
        """token: (B, 1, M) already embedded; pos: extended-sequence position.
        Returns mu, a for the token at index `pos` of the real sequence."""
        h = token + self.pos_embed.value[:, pos : pos + 1]  # (B, 1, M)
        for layer in self.layers:
            h = layer(h, pos=pos)  # decode path
        h = self.norm_out(h)
        mu, a = jnp.split(self.head(h), 2, axis=-1)  # (B, 1, D) each
        return mu[:, 0], a[:, 0]  # (B, D) each

    def inverse(
        self,
        z_next: jax.Array,
        y: jax.Array,
        y_null: jax.Array | None = None,
        guidance: float = 0.0,
    ) -> jax.Array:
        """z_next: (B, N, D) -> z: (B, N, D). O(N * K) with the KV cache.

        Classifier-free guidance (eq. 10): when `guidance > 0`, the batch is
        duplicated so that one half carries the real label and the other the
        null label, and the predicted affine parameters are extrapolated away
        from the unconditional prediction:

            mu_hat = mu_c + w * (mu_c - mu_u)
            a_hat  = a_c  + w * (a_c  - a_u)

        NOTE: verify this exact combination rule against eq. 10 of the paper
        before you trust a number you report. Appendix B also ramps the weight
        across blocks (w_t = t/(T-1) * w) rather than using it flat — that
        schedule lives in `TarFlow.sample`, not here.
        """
        B, N, D = z_next.shape
        use_cfg = guidance > 0.0 and y_null is not None
        batch = 2 * B if use_cfg else B
        labels = jnp.concatenate([y, y_null], axis=0) if use_cfg else y  # (batch,)

        self.init_cache(batch)

        # Extended position 0 is the conditioning token. Its output would be the
        # parameters for token 0, which eq. 3 defines as the identity, so we run
        # the step only to populate the cache and discard the result.
        c = self.class_embed(labels)[:, None, :]  # (batch, 1, M)
        self._decode_step(c, pos=0)

        z = jnp.zeros_like(z_next).at[:, 0].set(z_next[:, 0])  # token 0 identity

        for i in range(1, N):
            # Feed the token we just resolved (index i-1) at extended position i;
            # its output predicts the parameters for token i.
            prev = z[:, i - 1 : i]  # (B, 1, D)
            if use_cfg:
                prev = jnp.concatenate([prev, prev], axis=0)  # (2B, 1, D)
            mu, a = self._decode_step(self.proj_in(prev), pos=i)  # (batch, D)

            if use_cfg:
                mu_c, mu_u = jnp.split(mu, 2, axis=0)  # (B, D) each
                a_c, a_u = jnp.split(a, 2, axis=0)
                mu = mu_c + guidance * (mu_c - mu_u)
                a = a_c + guidance * (a_c - a_u)

            z = z.at[:, i].set(z_next[:, i] * jnp.exp(a) + mu)  # eq. 4

        return self._unpermute(z)


In [ ]:
# ---------------------------------------------------------------------------
# The full model
# ---------------------------------------------------------------------------


class TarFlow(nnx.Module):
    """Stack of T autoregressive flow blocks over a patchified image."""

    def __init__(self, cfg: TarFlowConfig, *, rngs: nnx.Rngs):
        self.cfg = cfg
        self.blocks = [ARFlowBlock(cfg, flow_index=t, rngs=rngs) for t in range(cfg.n_flows)]

    # -- x -> z_T -----------------------------------------------------------

    def forward(self, x: jax.Array, y: jax.Array) -> tuple[jax.Array, jax.Array]:
        """x: (B, C, H, W), y: (B,) -> (z_T: (B, N, D), total_logdet: (B,))."""
        z = patchify(x, self.cfg.patch_size)  # (B, N, D) = z^0
        logdet = jnp.zeros(x.shape[0], dtype=z.dtype)  # (B,)
        for block in self.blocks:  # t = 0 .. T-1
            z, ld = block.forward(z, y)
            logdet = logdet + ld
        return z, logdet  # z^T

    # -- loss (eq. 6) -------------------------------------------------------

    def nll(self, x: jax.Array, y: jax.Array) -> jax.Array:
        """Negative log-likelihood in nats per image. Shape (B,).

            -log p(x) = 0.5 * ||z^T||^2  +  sum_t sum_{i>=1} sum_j a^t_i[j]
                        + (N*D/2) * log(2*pi)

        The two data-dependent terms pull in opposite directions: the first
        wants small-norm latents, the second (via -log|det|) forbids collapsing
        the volume to get them. Log them SEPARATELY during training — if the
        total falls while sum(a) runs off to -inf you are collapsing the latent
        scale, not learning, and a scalar loss curve will not tell you.
        """
        z_T, logdet = self.forward(x, y)  # (B, N, D), (B,)
        ND = self.cfg.seq_len * self.cfg.token_dim
        gaussian = 0.5 * jnp.sum(z_T**2, axis=(1, 2))  # (B,)
        const = 0.5 * ND * math.log(2.0 * math.pi)
        return gaussian - logdet + const  # (B,)

    def bpd(self, x: jax.Array, y: jax.Array, data_range: float = 256.0) -> jax.Array:
        """Bits per dimension. Shape (B,).

        Assumes x has been scaled from [0, data_range) into whatever range the
        model sees; the log(data_range) term is the change-of-variables
        correction for that rescaling. Get this constant wrong and your BPD is
        wrong by a fixed offset — always sanity-check against a published number
        on the same dataset before reporting.
        """
        ND = self.cfg.seq_len * self.cfg.token_dim
        return self.nll(x, y) / (ND * math.log(2.0)) + math.log2(data_range)

    def loss(self, x: jax.Array, y: jax.Array, key: jax.Array) -> jax.Array:
        """Training loss: noise augmentation (section 2.4) + eq. 6, meaned."""
        # Additive Gaussian noise is what makes the density smooth enough for
        # the score-based denoising step below to be meaningful. Section 3.3
        # sweeps sigma; it trades likelihood against sample quality.
        x = x + self.cfg.noise_std * jax.random.normal(key, x.shape, x.dtype)
        return jnp.mean(self.nll(x, y))

    # -- score and Tweedie denoising (eqs. 7, 8) ----------------------------

    def score(self, x: jax.Array, y: jax.Array) -> jax.Array:
        """grad_x log p(x). Same shape as x.

        This is the payoff of exact likelihood: unlike a diffusion model, we do
        not train a separate score network. We differentiate the density the
        model already computes. One line of autodiff.
        """

        def logp(x_):
            return -jnp.sum(self.nll(x_, y))

        return jax.grad(logp)(x)

    def denoise(self, x: jax.Array, y: jax.Array) -> jax.Array:
        """Tweedie's formula (eq. 7):  E[x_clean | x] = x + sigma^2 * score(x).

        Samples come out of `sample` carrying the sigma-level noise the model
        was trained with; this projects them back toward the clean manifold and
        is most of the visual quality difference in the paper's figures.
        """
        return x + self.cfg.noise_std**2 * self.score(x, y)

    # -- z_T -> x -----------------------------------------------------------

    def sample(
        self,
        key: jax.Array,
        y: jax.Array,
        guidance: float = 0.0,
        schedule_guidance: bool = True,
        denoise: bool = True,
    ) -> jax.Array:
        """y: (B,) labels -> x: (B, C, H, W).

        Cost: T blocks x N sequential decode steps. For the paper's ImageNet 64
        config that is 8 x 1024 = 8192 sequential transformer calls, which is
        the ~2 minutes for 32 samples on an A100 reported in appendix D. The
        model is parallel in *time* (T is small) and sequential in *space*
        (N is large) — the mirror image of diffusion.
        """
        cfg = self.cfg
        B = y.shape[0]
        z = jax.random.normal(key, (B, cfg.seq_len, cfg.token_dim), cfg.dtype)  # z^T
        y_null = jnp.full_like(y, cfg.num_classes)  # the reserved null index

        # Blocks are inverted in reverse order: z^T -> z^{T-1} -> ... -> z^0.
        for t in reversed(range(cfg.n_flows)):
            if guidance > 0.0 and schedule_guidance and cfg.n_flows > 1:
                # Appendix B: ramp the weight across blocks rather than flat.
                w = guidance * t / (cfg.n_flows - 1)
            else:
                w = guidance
            z = self.blocks[t].inverse(z, y, y_null=y_null, guidance=w)

        x = unpatchify(z, cfg.patch_size, cfg.in_channels)  # (B, C, H, W)
        return self.denoise(x, y) if denoise else x

In [ ]:
# A small config so the shapes are readable; swap for the paper's
# "2-1024-8-8" to reproduce ImageNet 64x64.
cfg = TarFlowConfig(
    img_size=32,
    in_channels=3,
    patch_size=2,
    model_dim=128,
    n_heads=4,
    n_flows=4,
    n_layers=2,
    num_classes=10,
)
print(f"N = {cfg.seq_len}, D = {cfg.token_dim}")  # N = 256, D = 12

model = TarFlow(cfg, rngs=nnx.Rngs(0))
key = jax.random.key(0)

x = jax.random.normal(key, (2, 3, 32, 32))  # (B, C, H, W)
y = jnp.array([3, 7])  # (B,)

z_T, logdet = model.forward(x, y)  # (B, N, D), (B,)
print(z_T.shape, logdet.shape)

print(model.bpd(x, y).shape)  # (B,)
print(model.sample(key, y, guidance=1.5).shape)  # (B, C, H, W)

# The two checks to run before you ever start a real training job:
#
#   1. Invertibility, per block:
#        z1, _ = block.forward(z0, y)
#        assert jnp.allclose(block.inverse_naive(z1, y), z0, atol=1e-4)
#
#   2. Analytic log-det (eq. 5) against autodiff, on a tiny input:
#        f = lambda v: block.forward(v[None], y[:1])[0][0]
#        J = jax.jacfwd(f)(z0[0]).reshape(N * D, N * D)
#        _, logabsdet = jnp.linalg.slogdet(J)
#        assert jnp.allclose(logabsdet, logdet[0], atol=1e-4)
#
# Check 2 generalises: any time a paper claims a closed form for a
# determinant, divergence, trace or gradient, you can verify it against
# autodiff on a 3-dimensional toy in five lines.

N = 256, D = 12


ValueError: Found data on value of type '<class 'list'>' assigned to static attribute 'layers' of Pytree type '<class '__main__.ARFlowBlock'>'. Static attributes should not contain data values. Consider one of the following options:

1. If the attribute is meant to be static, remove the data values of type(s):

  TransformerLayer

2. If the attribute is meant to be data, wrap the value with nnx.data  on assignment:

  _.layers = nnx.data(...)

3. Alternatively, annotate the class attribute with nnx.data:

  class ARFlowBlock(Module):
    layers: list = nnx.data()

4. If the container is a list or dict, try using nnx.List(...) or nnx.Dict(...) instead.

5. Disable pytree for this class:

  class ARFlowBlock(Module, pytree=False):

